# Stage 2 — GRPO Self-Play in Pure Sim

**Goal:** Improve over the SFT baseline using on-environment RL (GRPO) in pure sim mode.  
**Runtime:** ~3 hours on Colab A100.  
**Prerequisites:** Stage 1 SFT checkpoint on HF Hub.

The env server must be running on a public URL (HF Space or ngrok).  
Set `ENV_URL` below before running.

In [ ]:
!pip install -q unsloth trl


In [ ]:
# Clone repo to get HeadlessRunner + env code
import os
if not os.path.exists('/content/AgentGrid_V1'):
    !git clone https://github.com/jayyyyqwq/GaN_J_AI.git /content/AgentGrid_V1
%cd /content/AgentGrid_V1
!pip install -q -e .

In [ ]:
import json, os, random, time
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────
SFT_CHECKPOINT = "Jayyyy234/agentgrid-sft"  # <- output of Stage 1
GRPO_REPO      = "Jayyyy234/agentgrid-grpo" # <- where to push result
# ENV_URL not needed — env runs in-process via HeadlessRunner
AGENTS         = ["A", "B", "C"]
EPISODE_STEPS  = 50
N_EPISODES     = 500   # total training episodes
GRPO_LR        = 1e-5
GRPO_EPOCHS    = 1
GROUP_SIZE     = 4     # GRPO group size
MAX_SEQ_LEN    = 2048
OUTPUT_DIR     = "/content/grpo_output"
LOG_PATH       = "/content/grpo_rewards.jsonl"
# ────────────────────────────────────────────────────────────────────

## Step 1: Load SFT model

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_CHECKPOINT,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=torch.float16,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    use_gradient_checkpointing=True,
)
print("SFT model loaded.")

## Step 2: Connect to env and run GRPO training loop

In [ ]:
# In-process GRPO loop using HeadlessRunner (no HTTP, no MCP server needed)
from agentgrid_spaces.runner import HeadlessRunner
from transformers import pipeline

FastLanguageModel.for_inference(model)
gen_pipeline = pipeline(
    'text-generation', model=model, tokenizer=tokenizer,
    max_new_tokens=120, do_sample=True, temperature=0.8,
)

TOOL_NAMES = ['broadcast', 'make_offer', 'accept_offer', 'execute_task', 'renege', 'idle']

def parse_tool_call(text: str) -> tuple[str, dict]:
    text = text.strip().strip('`').strip()
    if text.startswith('json'):
        text = text[4:].strip()
    try:
        obj = json.loads(text)
        tool = obj.get('tool', 'idle')
        kwargs = obj.get('kwargs', {})
        if tool not in TOOL_NAMES:
            return 'idle', {}
        return tool, kwargs
    except json.JSONDecodeError:
        return 'idle', {}

reward_log = []
runner = HeadlessRunner(episode_steps=EPISODE_STEPS)

for ep in range(N_EPISODES):
    snap = runner.reset(seed=ep)
    ep_rewards = {a: 0.0 for a in AGENTS}

    while not snap.done:
        for agent_id in AGENTS:
            obs = runner._env._format_observation(agent_id)
            prompt = (
                f'<s>[INST] {obs}\n'
                'Respond ONLY with a JSON object: {"tool": ..., "kwargs": {...}} [/INST]'
            )
            output = gen_pipeline(prompt)[0]['generated_text']
            completion = output[len(prompt):].strip()
            tool, kwargs = parse_tool_call(completion)
            runner.apply(agent_id, tool, **kwargs)
        snap = runner.snapshot()
        for a in AGENTS:
            ep_rewards[a] = snap.rewards[a]

    avg_return = sum(ep_rewards.values()) / len(AGENTS)
    promise_keep = snap.promise_keep_ratio
    reward_log.append({'episode': ep, 'avg_return': avg_return, 'promise_keep': promise_keep})
    if ep % 20 == 0:
        print(f'Episode {ep:4d}/{N_EPISODES}  avg_return={avg_return:.3f}  promise_keep={promise_keep:.2f}')
    with open(LOG_PATH, 'a') as f:
        f.write(json.dumps({'episode': ep, 'rewards': ep_rewards, 'promise_keep': promise_keep}) + '\n')

print('Training complete.')


## Step 3: Plot reward curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

episodes = [r["episode"] for r in reward_log]
returns  = [r["avg_return"] for r in reward_log]

# Smooth
window = 20
smoothed = np.convolve(returns, np.ones(window)/window, mode="valid")

plt.figure(figsize=(10, 5))
plt.plot(returns, alpha=0.3, label="Raw return")
plt.plot(range(window-1, len(returns)), smoothed, label=f"Smoothed (w={window})")
plt.xlabel("Episode")
plt.ylabel("Avg episode return (3 agents)")
plt.title("GRPO Self-Play — AgentGrid Sim")
plt.legend()
plt.tight_layout()
plt.savefig("/content/grpo_curve.png", dpi=150)
plt.show()
print("Saved: /content/grpo_curve.png")

## Step 4: Push GRPO checkpoint to HF Hub

In [ ]:
from huggingface_hub import login
login()

model.push_to_hub(GRPO_REPO)
tokenizer.push_to_hub(GRPO_REPO)
print(f"GRPO checkpoint saved to hf.co/{GRPO_REPO}")